# 10 - Salary Prediction

**Goal**: Train a regression model that predicts `med_salary_yearly` from job description features.

**Research Theme 1**: Can NLP features from job descriptions (sentiment, senior signals, description length)
combined with structured features (experience level, remote status, skill count) predict salary?

**Challenge**: Only ~5% of postings (6,280 rows) have salary labels. We train on this labeled subset.

**Pipeline**:
```
postings_features.parquet
  → filter to rows with med_salary_yearly
  → encode categoricals
  → Ridge baseline → XGBoost → SHAP feature importance
  → save best model
```

**Learning concepts**: train/test split, cross-validation, RMSE vs MAE, feature importance with SHAP,
log-transforming skewed targets.

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from loguru import logger
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

from talentlens.config import (
    POSTINGS_FEATURES_PARQUET,
    RANDOM_SEED,
    SALARY_MODEL_PATH,
)
from talentlens.plots import save_fig

pd.set_option('display.max_columns', None)
%matplotlib inline

# ── Cache control ──────────────────────────────────────────────────────────────
FORCE_RECOMPUTE = False

logger.info("Libraries loaded.")

## 1. Load Data & Filter to Labeled Rows

In [ ]:
df = pd.read_parquet(POSTINGS_FEATURES_PARQUET)
logger.info(f"Loaded {len(df):,} postings")

# Keep only rows with a salary label
df_salary = df[df['med_salary_yearly'].notna()].copy()
logger.info(f"Rows with salary: {len(df_salary):,} ({len(df_salary)/len(df)*100:.1f}% of all postings)")

# Remove obvious outliers: salary < $10K or > $1M (likely data errors)
df_salary = df_salary[
    df_salary['med_salary_yearly'].between(10_000, 1_000_000)
].copy()
logger.info(f"After outlier removal: {len(df_salary):,} rows")

print("\nSalary distribution:")
print(df_salary['med_salary_yearly'].describe().apply(lambda x: f"${x:,.0f}"))

In [ ]:
# Log-transform the target: salary is right-skewed, log makes it more normal
# This improves linear model performance and reduces leverage of very high salaries.
df_salary['log_salary'] = np.log1p(df_salary['med_salary_yearly'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(df_salary['med_salary_yearly'], bins=50, color='steelblue')
axes[0].set_title('Raw Salary Distribution (right-skewed)')
axes[0].set_xlabel('Median Annual Salary ($)')

axes[1].hist(df_salary['log_salary'], bins=50, color='coral')
axes[1].set_title('Log(Salary) Distribution (more normal)')
axes[1].set_xlabel('log(1 + Salary)')

plt.tight_layout()
save_fig(fig, 'salary_distribution')
plt.show()

## 2. Feature Engineering

We use only features that are available at posting time (no leakage):
- **Numeric**: desc_word_count, sentiment_polarity, senior_signal_count, max_years_required, n_skills
- **Categorical**: experience_level, is_remote (encoded)
- **No embeddings here**: embeddings are 384-dim and we only have 6K rows → high risk of overfitting.
  We'll use interpretable features only.

In [ ]:
NUMERIC_FEATURES = [
    'desc_word_count',
    'desc_char_count',
    'title_word_count',
    'sentiment_polarity',
    'sentiment_subjectivity',
    'senior_signal_count',
    'max_years_required',
    'n_skills',
]

CATEGORICAL_FEATURES = ['experience_level', 'is_remote']

TARGET = 'log_salary'

# Encode experience_level ordinally (intern < entry < associate < mid-senior < director < executive)
exp_order = {
    'Internship': 0, 'Entry level': 1, 'Associate': 2,
    'Mid-Senior level': 3, 'Director': 4, 'Executive': 5, 'Unknown': 2
}
df_salary['exp_level_encoded'] = df_salary['experience_level'].map(exp_order).fillna(2)
df_salary['is_remote_int'] = df_salary['is_remote'].astype(int)

FEATURE_COLS = NUMERIC_FEATURES + ['exp_level_encoded', 'is_remote_int']

X = df_salary[FEATURE_COLS].fillna(0).values
y = df_salary[TARGET].values

print(f"Feature matrix: {X.shape}")
print(f"Target vector: {y.shape}")
print(f"\nFeature columns: {FEATURE_COLS}")

## 3. Baseline: Ridge Regression

**Why Ridge?** It's fast, interpretable, and gives us a baseline to beat.
Ridge adds L2 regularization (penalizes large coefficients), which prevents overfitting
when features are correlated.

**Why cross-validation?** With only 6K rows, a single train/test split is noisy.
5-fold CV gives a more stable estimate of generalization performance.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_SEED
)

# Ridge pipeline: scale features first (Ridge is sensitive to feature scale)
ridge_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('ridge', Ridge(alpha=10.0)),
])

# 5-fold cross-validation on training set
kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_scores = cross_val_score(ridge_pipe, X_train, y_train, cv=kf, scoring='r2')

print(f"Ridge 5-fold CV R²: {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")
print(f"  Individual folds: {cv_scores.round(3)}")

# Fit on full training set and evaluate on test set
ridge_pipe.fit(X_train, y_train)
y_pred_ridge = ridge_pipe.predict(X_test)

# Convert log-salary predictions back to dollars for interpretable RMSE
y_pred_dollars = np.expm1(y_pred_ridge)
y_test_dollars = np.expm1(y_test)

rmse = np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars))
mae = mean_absolute_error(y_test_dollars, y_pred_dollars)
r2 = r2_score(y_test, y_pred_ridge)

print(f"\nRidge Test Set Performance:")
print(f"  R²:   {r2:.3f}")
print(f"  RMSE: ${rmse:,.0f}")
print(f"  MAE:  ${mae:,.0f}")

## 4. XGBoost — Better Performance

**Why XGBoost over Ridge?**
- Captures non-linear relationships (salary may not scale linearly with years mentioned)
- Automatically handles feature interactions (e.g., senior + remote combined)
- Built-in feature importance scores

**Why not deep learning?** Only ~5K training rows. Neural networks need much more data.
XGBoost is typically the best choice for tabular data at this scale.

In [ ]:
if not FORCE_RECOMPUTE and SALARY_MODEL_PATH.exists():
    logger.info(f"Loading cached model from {SALARY_MODEL_PATH}")
    best_model = joblib.load(SALARY_MODEL_PATH)
else:
    try:
        from xgboost import XGBRegressor
        xgb = XGBRegressor(
            n_estimators=500,
            max_depth=4,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=RANDOM_SEED,
            verbosity=0,
        )
        cv_xgb = cross_val_score(xgb, X_train, y_train, cv=kf, scoring='r2')
        logger.info(f"XGBoost 5-fold CV R²: {cv_xgb.mean():.3f} ± {cv_xgb.std():.3f}")

        xgb.fit(X_train, y_train)
        best_model = xgb
    except ImportError:
        logger.warning("XGBoost not installed — using Ridge as best model. `pip install xgboost`")
        best_model = ridge_pipe.fit(X_train, y_train)

    SALARY_MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
    joblib.dump(best_model, SALARY_MODEL_PATH)
    logger.info(f"Model saved to {SALARY_MODEL_PATH}")

# Evaluate best model on test set
y_pred_best = best_model.predict(X_test)
y_pred_dollars_best = np.expm1(y_pred_best)

rmse_best = np.sqrt(mean_squared_error(y_test_dollars, y_pred_dollars_best))
mae_best = mean_absolute_error(y_test_dollars, y_pred_dollars_best)
r2_best = r2_score(y_test, y_pred_best)

print(f"Best Model Test Set Performance:")
print(f"  R²:   {r2_best:.3f}")
print(f"  RMSE: ${rmse_best:,.0f}")
print(f"  MAE:  ${mae_best:,.0f}")
print(f"\nBaseline (Ridge): R² = {r2:.3f}")
print(f"Improvement: +{r2_best - r2:.3f} R²")

## 5. SHAP Feature Importance

**Why SHAP?** Standard feature importances from tree models can be misleading
(they favor high-cardinality features). SHAP (SHapley Additive exPlanations) gives each
feature a contribution score based on game theory — it's the most reliable way to
understand *why* a model makes a prediction.

**What we're looking for**: Which features most influence predicted salary?
If `exp_level_encoded` and `senior_signal_count` dominate, it confirms that seniority
language in job descriptions is a reliable salary predictor.

In [ ]:
try:
    import shap
    from xgboost import XGBRegressor

    if isinstance(best_model, XGBRegressor):
        explainer = shap.TreeExplainer(best_model)
        shap_values = explainer.shap_values(X_test)

        feature_names_display = [
            'desc_word_count', 'desc_char_count', 'title_word_count',
            'sentiment_polarity', 'sentiment_subjectivity',
            'senior_signals', 'max_years_req', 'n_skills',
            'exp_level', 'is_remote'
        ]

        fig, ax = plt.subplots(figsize=(8, 5))
        shap.summary_plot(
            shap_values, X_test,
            feature_names=feature_names_display,
            plot_type='bar',
            show=False
        )
        ax = plt.gca()
        ax.set_title('SHAP Feature Importance — Salary Prediction')
        save_fig(plt.gcf(), 'salary_shap_importance')
        plt.show()

        # Also show beeswarm: direction of effect
        shap.summary_plot(
            shap_values, X_test,
            feature_names=feature_names_display,
            show=False
        )
        save_fig(plt.gcf(), 'salary_shap_beeswarm')
        plt.show()
    else:
        # Ridge: use coefficients as proxy for importance
        coefs = dict(zip(FEATURE_COLS, best_model.named_steps['ridge'].coef_))
        coef_df = pd.DataFrame.from_dict(coefs, orient='index', columns=['coefficient'])
        coef_df['abs_coef'] = coef_df['coefficient'].abs()
        coef_df = coef_df.sort_values('abs_coef', ascending=False)
        print("Ridge Coefficients (proxy for importance):")
        print(coef_df.round(4))
except ImportError:
    logger.warning("SHAP not installed. `pip install shap` to see feature importance plots.")

## 6. Prediction vs Actual Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test_dollars / 1000, y_pred_dollars_best / 1000, alpha=0.3, s=15, color='steelblue')
lims = [0, max(y_test_dollars.max(), y_pred_dollars_best.max()) / 1000]
ax.plot(lims, lims, 'r--', alpha=0.8, label='Perfect prediction')
ax.set_xlabel('Actual Salary ($K)')
ax.set_ylabel('Predicted Salary ($K)')
ax.set_title(f'Salary Prediction — Actual vs Predicted\nR² = {r2_best:.3f}, MAE = ${mae_best/1000:.1f}K')
ax.legend()
save_fig(fig, 'salary_prediction_scatter')
plt.show()

## Summary

### What we built
- **Ridge baseline**: L2-regularized linear regression on normalized features
- **XGBoost**: Gradient-boosted tree ensemble (typically best for tabular data)
- **SHAP**: Feature contribution analysis to understand *which* features drive salary predictions

### Key insight
The model tells us **which features correlate with salary**: experience level, senior language,
remote status, and description length. This is the quantitative answer to Research Theme 1.

### Limitations
- Only 5% of postings have salary data — we can't predict salary for the other 95%
- Selection bias: jobs that disclose salary may be systematically different
- No company-level features (company size, industry) — would likely improve predictions

---
**→ Next**: Notebook 11 — Ghost Job Classifier